In [1]:
import pandas as pd
import requests
from pathlib import Path
from time import sleep

In [2]:
BASE_URL = "https://data.ademe.fr/data-fair/api/v1/datasets/dpe03existant/lines"
PAGE_SIZE = 10000
TARGET_ROWS = 50000

QS_FILTER = 'type_batiment:"maison" OR type_batiment:"appartement"'

def fetch_logements(target_rows=TARGET_ROWS, page_size=PAGE_SIZE, qs=QS_FILTER):
    params = {"size": page_size, "qs": qs}
    url = BASE_URL
    pages = []
    fetched = 0
    while url and fetched < target_rows:
        r = requests.get(url, params=params if url == BASE_URL else None, timeout=60)
        r.raise_for_status()
        payload = r.json()
        results = payload.get("results", [])
        if not results:
            break
        pages.append(pd.DataFrame(results))
        fetched += len(results)
        print(f"fetched {fetched:>7} / {target_rows} (total available: {payload.get('total')})")
        url = payload.get("next")
        sleep(0.2)
    return pd.concat(pages, ignore_index=True).head(target_rows)

df = fetch_logements()
df.shape

fetched   10000 / 50000 (total available: 14345566)
fetched   20000 / 50000 (total available: None)
fetched   30000 / 50000 (total available: None)
fetched   40000 / 50000 (total available: None)
fetched   50000 / 50000 (total available: None)


(50000, 220)

In [4]:
print("rows:", len(df))
print("cols:", df.shape[1])
print("\ntype_batiment counts:")
print(df["type_batiment"].value_counts(dropna=False))

rows: 50000
cols: 220

type_batiment counts:
type_batiment
appartement    33544
maison         16456
Name: count, dtype: int64


In [5]:
out_dir = Path("../data/raw")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / f"dpe_logements_{len(df)}.parquet"
df.to_parquet(out_path, index=False)
print(f"saved {out_path} ({out_path.stat().st_size / 1e6:.1f} MB)")

saved ../data/raw/dpe_logements_50000.parquet (23.9 MB)


In [ ]:
TARGET = "periode_construction"
ANNEE_CONSTRUCTION = "annee_construction"  # target for regression 

# Energy features 
ENERGY_FEATURES = [
    "conso_5_usages_par_m2_ep",        # kWh/m²/year — standardised total
    "emission_ges_5_usages_par_m2",    # kg CO₂/m²/year
    "etiquette_dpe",                   # A-G label (ordinal)
    "etiquette_ges",                   # A-G label (ordinal)
    "conso_chauffage_ep",              # kWh/m²/year heating
    "conso_ecs_ep",                    # kWh/m²/year hot water
    "conso_refroidissement_ep",        # kWh/m²/year cooling
    "conso_eclairage_ep",              # kWh/m²/year lighting (post-2021 only)
    "conso_auxiliaires_ep",            # kWh/m²/year auxiliaries (post-2021 only)
]

# Structural features 
STRUCTURAL_FEATURES = [
    "surface_habitable_logement",      # m² habitable surface
    "type_batiment",                   # maison vs. appartement
    "nombre_niveau_logement",          # number of floors
    "qualite_isolation_murs",          # wall insulation type/quality
    "qualite_isolation_menuiseries",   # window glazing type
    "type_generateur_chauffage_principal",  # heating system type (boiler, heat pump, etc.)
    "type_energie_principale_chauffage",    # heating energy (gas, electric, oil, etc.)
    "type_installation_ecs",           # hot water system type
    "type_energie_principale_ecs",     # hot water energy type
    "type_ventilation",                # natural vs. mechanical (VMC)
    "isolation_toiture",               # roof insulation type/quality
    "qualite_isolation_plancher_bas",  # floor insulation type/quality
    "classe_inertie_batiment",         # thermal mass / inertia class
]

# Geographic features
GEO_FEATURES = [
    "code_departement_ban",   # French département code
    "zone_climatique",        # climate zone (H1, H2, H3)
    "classe_altitude",        # altitude category
]

ALL_FEATURES = ENERGY_FEATURES + STRUCTURAL_FEATURES + GEO_FEATURES
print(f"Target: {TARGET}")
print(f"Features to keep: {len(ALL_FEATURES)} ({len(ENERGY_FEATURES)} energy, {len(STRUCTURAL_FEATURES)} structural, {len(GEO_FEATURES)} geo)")

# Verify all feature columns exist
missing = [c for c in ALL_FEATURES if c not in df.columns]
if missing:
    print(f"WARNING: Missing columns: {missing}")
else:
    print(f"All feature columns present in data")

# Select target and features only
df_clean = df[[TARGET] + ALL_FEATURES].copy()
print(f"\nShape after feature selection: {df_clean.shape}")
print(f"Removed {df.shape[1] - df_clean.shape[1]} administrative/ID/address columns")

Target: periode_construction
Features to keep: 25 (9 energy, 13 structural, 3 geo)
✓ All feature columns present in data

Shape after feature selection: (50000, 26)
Removed 194 administrative/ID/address columns
